## Prospect Data Ingestion Notebook

This notebook fetches draft prospect data from the FanGraphs API and writes it to a Unity Catalog-backed volume. The saved JSON file serves as the raw input for the bronze layer of the Delta Live Tables pipeline.

It uses parameters like `board_type`, `season`, and `source` to construct the API request and generate a partitioned output path based on ingest date. The structure supports downstream AutoLoader ingestion and medallion-style organization.

Run this notebook before triggering the DLT pipeline to ensure the latest data is available.

Data Source: https://www.fangraphs.com/prospects/the-board/2025-mlb-draft/summary

In [0]:
import requests
import json
from datetime import date
import os

In [0]:
# ---- Parameters ----
board_type = "mlb" # Options: "prospect" (preseason), "updated" (midseason), "mlb" (draft), "int" 
season = 2025
source = "fangraphs"
ingest_date = date.today().isoformat()
draft = f"{season}{board_type}"

# ---- API Call ----
url = f"https://www.fangraphs.com/api/prospects/board/data?draft={draft}&season={season}"
response = requests.get(url)
response.raise_for_status()
data = response.json()

# ---- Build Volume File Path ----
base_path = "/Volumes/alexander_booth/rag_demo/prospect_data"
output_dir = f"{base_path}/source={source}/board_type={board_type}/season={season}/ingest_date={ingest_date}"
os.makedirs(output_dir, exist_ok=True)  # Ensure directories exist

output_path = f"{output_dir}/fangraphs_prospects.json"

# ---- Save Raw JSON ----
with open(output_path, "w") as f:
    json.dump(data, f, indent=2)

print(f"✅ Raw JSON saved to: {output_path}")


✅ Raw JSON saved to: /Volumes/alexander_booth/rag_demo/prospect_data/source=fangraphs/board_type=mlb/season=2025/ingest_date=2025-10-28/fangraphs_prospects.json
